# Pre-entrega 6 · Orquestador multi-agente

Demo ejecutada de **investigación → análisis → síntesis**, con una segunda
ejecución que requiere refinar una búsqueda sin evidencia.

**Qué es real:** el grafo jerárquico de `orchestrator/graph.py`, el estado
compartido, las herramientas de búsqueda y sentimiento, y el corpus local.
**Qué se simula:** las respuestas de los tres modelos y la recuperación
(búsqueda exacta sobre un fragmento del corpus, sin índice vectorial).
Estos dobles hacen la demo reproducible sin claves; no prueban la autonomía
de un modelo real. Para esa modalidad usa `python demo_orchestrator.py`
con OpenRouter y el índice Chroma, como explica el README.

## 1. Preparación

Desde la raíz del repositorio instala `requirements-notebook.txt` en tu entorno
y selecciona ese Python como kernel. Ejecuta todas las celdas en orden.

In [1]:
import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.runnables import RunnableLambda

from orchestrator.agents.analyst_agent import score_sentiment
from orchestrator.graph import build_orchestrator_graph

source = Path("data/03_uso_gastos_seguridad.txt")
corpus = source.read_text(encoding="utf-8")
fragment = corpus.split("Artículo 32.-", 1)[1].split("Igual mérito", 1)[0]
fragment = "Artículo 32.- " + " ".join(fragment.split())
document = Document(page_content=fragment, metadata={"source": source.as_posix()})
request = (
    "Busca el artículo 32 sobre cobro de gastos comunes "
    "y analiza el tono del fragmento."
)
print("Solicitud:", request)
print("Fuente local:", source)
print("Fragmento:", fragment)

Solicitud: Busca el artículo 32 sobre cobro de gastos comunes y analiza el tono del fragmento.
Fuente local: data/03_uso_gastos_seguridad.txt
Fragmento: Artículo 32.- Los avisos de cobro de los gastos comunes y de las demás obligaciones económicas adeudadas por los copropietarios, siempre que se encuentren firmados de forma presencial o electrónica por el administrador, tendrán mérito ejecutivo para el cobro de los mismos.


## 2. Modelos de demostración

El doble del Supervisor inspecciona los aportes recibidos: si no hay evidencia,
delega o pide refinar; con evidencia la envía al analista; con ambos resultados
sintetiza. Esta política determinista vive solamente en el notebook. El
Supervisor del prototipo sigue delegando mediante la decisión JSON del LLM.

Los dobles de especialistas emiten llamadas que `ToolNode` ejecuta. Sus
respuestas se construyen a partir de los resultados de las herramientas.

In [2]:
class DemoSupervisor:
    def __init__(self, refine=False):
        self.refine = refine

    async def ainvoke(self, messages):
        lines = str(messages[-1].content).splitlines()
        research = [
            json.loads(line.removeprefix("- researcher: "))
            for line in lines
            if line.startswith("- researcher: ")
        ]
        analysis = [
            json.loads(line.removeprefix("- analyst: "))
            for line in lines
            if line.startswith("- analyst: ")
        ]
        evidence = [item for item in research if item["status"] == "ok"]
        if not research:
            query = "consulta general" if self.refine else "artículo 32"
            decision = {"next": "researcher", "instruction": query}
        elif not evidence:
            decision = {
                "next": "researcher",
                "instruction": "Falta evidencia y fuente: busca artículo 32",
            }
        elif not analysis:
            result = evidence[-1]["resultados"][0]
            decision = {
                "next": "analyst",
                "instruction": json.dumps(
                    {"texto": result["fragmento"], "fuente": result["fuente"]},
                    ensure_ascii=False,
                ),
            }
        else:
            result = evidence[-1]["resultados"][0]
            sentiment = analysis[-1]
            answer = (
                f"{result['fragmento']} Fuente: {result['fuente']}. "
                f"Tono léxico: {sentiment['etiqueta']} "
                f"con puntaje {sentiment['puntaje']}."
            )
            decision = {"next": "end", "instruction": answer}
        return AIMessage(content=json.dumps(decision, ensure_ascii=False))

In [3]:
class DemoSpecialist:
    def __init__(self, role, tool_events):
        self.role = role
        self.tool_events = tool_events

    def bind_tools(self, tools):
        available = {tool.name for tool in tools}

        async def respond(messages):
            observations = [m for m in messages if isinstance(m, ToolMessage)]
            if observations:
                # Devuelve el resultado observado, no una respuesta prefijada.
                result = json.loads(observations[-1].content)
                serialized = json.dumps(result, ensure_ascii=False)
                print(f"  Resultado {self.role}: {serialized}")
                return AIMessage(content=json.dumps(result, ensure_ascii=False))
            assert len(messages) == 2  # Solo sistema e instrucción del Supervisor.
            instruction = str(messages[1].content)
            if self.role == "researcher":
                name = "buscar_en_base_conocimiento"
                args = {"consulta": instruction}
            else:
                name = "analizar_sentimiento"
                args = {"texto": json.loads(instruction)["texto"]}
            assert name in available
            self.tool_events.append({"agent": self.role, "tool": name, "args": args})
            print(f"  Herramienta {self.role}: {name}({args})")
            return AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": name,
                        "args": args,
                        "id": f"call-{len(self.tool_events)}",
                        "type": "tool_call",
                    }
                ],
            )

        return RunnableLambda(respond)


async def local_lookup(query):
    # Simula un retriever de un fragmento. La consulta general no tiene resultados.
    return [document] if "artículo 32" in query.casefold() else []

## 3. Ejecutar el grafo y observar las delegaciones

Cada evento muestra qué nodo intervino. El estado acumula los aportes del
investigador y del analista; el último mensaje del Supervisor es la síntesis.
El analista recibe el texto recuperado, sin el historial interno del investigador.

In [4]:
async def demonstrate(refine=False):
    tool_events = []
    graph = build_orchestrator_graph(
        supervisor_model=DemoSupervisor(refine=refine),
        researcher_model=DemoSpecialist("researcher", tool_events),
        analyst_model=DemoSpecialist("analyst", tool_events),
        retriever=RunnableLambda(local_lookup),
        max_steps=6,
    )
    state = {
        "messages": [HumanMessage(content=request)],
        "contributions": [],
        "next_agent": "researcher",
        "instruction": request,
        "task_completed": False,
        "step_count": 0,
    }
    route = []
    final = None
    async for mode, event in graph.astream(state, stream_mode=["updates", "values"]):
        if mode == "values":
            final = event
            continue
        for node, update in event.items():
            route.append(node)
            if node == "supervisor":
                print(f"Supervisor #{update['step_count']} → {update['next_agent']}")
            else:
                print(f"{node} → Supervisor (aporte conservado)")
    print()
    print("Recorrido:", " → ".join(["START", *route, "END"]))
    print("Síntesis:", final["messages"][-1].content)
    print("Aportes:", [item["agent"] for item in final["contributions"]])
    return final, route, tool_events


normal, normal_route, normal_tools = await demonstrate()

Supervisor #1 → researcher
  Herramienta researcher: buscar_en_base_conocimiento({'consulta': 'artículo 32'})
  Resultado researcher: {"status": "ok", "consulta": "artículo 32", "resultados": [{"fuente": "data/03_uso_gastos_seguridad.txt", "fragmento": "Artículo 32.- Los avisos de cobro de los gastos comunes y de las demás obligaciones económicas adeudadas por los copropietarios, siempre que se encuentren firmados de forma presencial o electrónica por el administrador, tendrán mérito ejecutivo para el cobro de los mismos."}]}
researcher → Supervisor (aporte conservado)
Supervisor #2 → analyst
  Herramienta analyst: analizar_sentimiento({'texto': 'Artículo 32.- Los avisos de cobro de los gastos comunes y de las demás obligaciones económicas adeudadas por los copropietarios, siempre que se encuentren firmados de forma presencial o electrónica por el administrador, tendrán mérito ejecutivo para el cobro de los mismos.'})
  Resultado analyst: {"status": "ok", "etiqueta": "neutral", "puntaj

## 4. Refinar cuando la búsqueda inicial no aporta evidencia

En este escenario controlado, la primera consulta no encuentra el fragmento.
El Supervisor solicita el artículo concreto y su fuente antes de delegar al
analista. El aporte sin resultados permanece en el estado junto al aporte
corregido para conservar la trazabilidad.

In [5]:
refined, refined_route, refined_tools = await demonstrate(refine=True)

Supervisor #1 → researcher
  Herramienta researcher: buscar_en_base_conocimiento({'consulta': 'consulta general'})
  Resultado researcher: {"status": "not_found", "consulta": "consulta general", "resultados": []}
researcher → Supervisor (aporte conservado)
Supervisor #2 → researcher
  Herramienta researcher: buscar_en_base_conocimiento({'consulta': 'Falta evidencia y fuente: busca artículo 32'})
  Resultado researcher: {"status": "ok", "consulta": "Falta evidencia y fuente: busca artículo 32", "resultados": [{"fuente": "data/03_uso_gastos_seguridad.txt", "fragmento": "Artículo 32.- Los avisos de cobro de los gastos comunes y de las demás obligaciones económicas adeudadas por los copropietarios, siempre que se encuentren firmados de forma presencial o electrónica por el administrador, tendrán mérito ejecutivo para el cobro de los mismos."}]}
researcher → Supervisor (aporte conservado)
Supervisor #3 → analyst
  Herramienta analyst: analizar_sentimiento({'texto': 'Artículo 32.- Los avisos

## 5. Comprobaciones del flujo

Se verifica el recorrido, el uso real de las herramientas, la conservación del
aporte incompleto y que el analista haya procesado exactamente la evidencia
recuperada. La etiqueta de sentimiento es una clasificación léxica sencilla;
no representa una interpretación jurídica del artículo.

In [6]:
assert normal_route == [
    "supervisor",
    "researcher",
    "supervisor",
    "analyst",
    "supervisor",
]
assert refined_route == [
    "supervisor",
    "researcher",
    "supervisor",
    "researcher",
    "supervisor",
    "analyst",
    "supervisor",
]
assert normal["task_completed"] and refined["task_completed"]
assert normal["step_count"] == 3 and refined["step_count"] == 4
assert len(normal["contributions"]) == 2 and len(refined["contributions"]) == 3
assert json.loads(refined["contributions"][0]["summary"])["status"] == "not_found"
for result, events in [(normal, normal_tools), (refined, refined_tools)]:
    evidence = next(
        json.loads(c["summary"])
        for c in result["contributions"]
        if c["agent"] == "researcher" and json.loads(c["summary"])["status"] == "ok"
    )
    recovered = evidence["resultados"][0]["fragmento"]
    assert events[-1]["tool"] == "analizar_sentimiento"
    assert events[-1]["args"]["texto"] == recovered == fragment
    assert json.loads(result["contributions"][-1]["summary"]) == score_sentiment(
        recovered
    )
    assert source.as_posix() in result["messages"][-1].content
print("OK: delegación, herramientas, refinamiento, contexto y síntesis verificados.")

OK: delegación, herramientas, refinamiento, contexto y síntesis verificados.


La suite `python -m pytest -q tests/test_orchestrator.py` cubre además la
parada por límite de pasos y los reintentos ante decisiones inválidas.
La política de conflictos y las instrucciones para la ejecución con un modelo
real están documentadas en el README.